# Loan Default Prediction Implementation
This notebook replicates the COMPLETE ML pipeline from `Loan main.ipynb`, including full preprocessing, model comparison, overfitting checks, and API logic.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, 
    roc_curve, auc
)

import warnings
warnings.filterwarnings('ignore')

In [2]:
# 1. Load Data
df = pd.read_csv('Loan_default.csv')
print(f"Initial data shape: {df.shape}")
df.head()

Initial data shape: (255347, 18)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [3]:
# 2. Preprocessing - Outlier Removal using IQR
# Selecting key numeric columns for outlier check based on source pattern
cols_to_check = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']

for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

print(f"Data shape after outlier removal: {df.shape}")

Data shape after outlier removal: (255347, 18)


In [4]:
# 3. Preprocessing - Feature Encoding
# Handle binary features (mapping as seen in source)
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Target encoding (Default is numeric already usually, but let's ensure it's clean)
y = df['Default']
X = df.drop(columns=['Default', 'LoanID'])

# Define categorical columns for One-Hot Encoding via ColumnTransformer
cat_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']

ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(drop='first'), cat_cols)
    ],
    remainder='passthrough'
)

X_encoded = ct.fit_transform(X)
print(f"Encoded features shape: {X_encoded.shape}")

Encoded features shape: (255347, 24)


In [5]:
# 4. Data Splitting and Scaling
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

print("Data Split and Scaling Complete.")

Data Split and Scaling Complete.


In [ ]:
# 5. Multi-Model Comparison (5-Fold Cross-Validation)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=10),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Gradient Boosting": GradientBoostingClassifier()
}

print("--- Model Performance (5-Fold Cross-Validation) ---")
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    print(f"{name} Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
# 6. Final Model & Overfitting Check
final_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
final_model.fit(X_train_scaled, y_train)

train_acc = accuracy_score(y_train, final_model.predict(X_train_scaled))
test_acc = accuracy_score(y_test, final_model.predict(X_test_scaled))

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy:  {test_acc:.4f}")

if train_acc > test_acc + 0.05:
    print("Result: Model is Overfitting.")
elif train_acc < 0.70:
    print("Result: Model is Underfitting.")
else:
    print("Result: Model is Generalized (Appropriate fit).")

In [ ]:
# 7. Evaluation & Visualization
y_preds = final_model.predict(X_test_scaled)
y_proba = final_model.predict_proba(X_test_scaled)[:, 1]
print("\nClassification Report:")
print(classification_report(y_test, y_preds))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_preds), annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Final Model")
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
# 8. Model Export & API Logic
os.makedirs("loan_api", exist_ok=True)
joblib.dump(final_model, "loan_api/model.pkl")
joblib.dump(scaler, "loan_api/scaler.pkl")
joblib.dump(ct, "loan_api/encoder.pkl")
print("Model components saved to loan_api/")

In [ ]:
%%writefile loan_api/app.py
from flask import Flask, request, jsonify
import joblib
import numpy as np
import pandas as pd

app = Flask(__name__)

model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")
encoder = joblib.load("encoder.pkl")

@app.route("/")
def home():
    return "Loan Default Prediction API is Running!"

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    df_input = pd.DataFrame([data])
    
    # Preprocess matching the training pipeline
    encoded = encoder.transform(df_input)
    scaled = scaler.transform(encoded)
    
    pred = model.predict(scaled)[0]
    proba = model.predict_proba(scaled)[0][1]
    
    return jsonify({
        "prediction": int(pred),
        "default_probability": float(proba)
    })

if __name__ == "__main__":
    app.run(debug=True)